In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.ML.Descriptors import MoleculeDescriptors
from rdkit.Chem import Descriptors
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
from scipy.stats import pearsonr
from scipy.stats import spearmanr
from scipy.stats import rankdata
import matplotlib.pyplot as plt
import seaborn as sns
import json

import warnings
from tqdm import tqdm
import os
from io import StringIO
import datetime
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem, DataStructs
from rdkit.ML.Cluster import Butina
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem import AllChem
from rdkit import Chem
from rdkit.DataStructs import BulkTanimotoSimilarity, ConvertToNumpyArray
from rdkit.DataStructs.cDataStructs import TanimotoSimilarity
from rdkit import RDLogger
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import Crippen
from rdkit.Chem import Draw
import sys
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
from scipy.stats import spearmanr
from scipy.stats import rankdata
import itertools
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge
import re
import glob
from collections import Counter

from collections import defaultdict
import random
import xgboost as xgb
from xgboost import XGBRegressor
import joblib
import pickle
import optuna

print(dir(xgb))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# ------------------------------
# Global seaborn style
# ------------------------------
sns.set_theme(
    context="notebook",
    style="whitegrid",     # adds a light grid to the background
    palette="viridis",     # use viridis for all seaborn plots
    font_scale=1.2         # ~16pt font for labels/ticks
)

# ------------------------------
# Global matplotlib configuration
# ------------------------------
plt.rcParams.update({

    # --- Font / text ---
    "font.size": 16,                 # base font size
    "axes.titlesize": 16,
    "axes.labelsize": 16,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 10,

    # --- Colormap ---
    "image.cmap": "viridis",

    # --- Axes appearance ---
    "axes.grid": True,               # always show grid
    "grid.alpha": 0.4,
    "grid.linestyle": "--",

    # --- Ticks ---
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.major.size": 6,
    "ytick.major.size": 6,
    "xtick.minor.size": 3,
    "ytick.minor.size": 3,

    # --- Spines (frame) ---
    "axes.spines.top": True,
    "axes.spines.right": True,       # ensures frame is closed
    "axes.spines.left": True,
    "axes.spines.bottom": True,

    # --- Legend ---
    "legend.frameon": True,          # legend box visible
    "legend.framealpha": 0.8,
    "legend.borderpad": 0.1,
    "legend.labelspacing": 0.2, 


    # --- Save format ---
    "savefig.dpi": 600
})

print("✅ Global plotting style configured.")


In [ ]:
def optimise(df, feature_cols, target_col, split, outname,
             draw_plots=False, n_trials=30):

    # =============================
    # Split into sets
    # =============================
    X_train = df.loc[split == "train", feature_cols].reset_index(drop=True)
    X_val   = df.loc[split == "val", feature_cols].reset_index(drop=True)
    X_test  = df.loc[split == "test", feature_cols].reset_index(drop=True)

    y_train = df.loc[split == "train", target_col].reset_index(drop=True)
    y_val   = df.loc[split == "val", target_col].reset_index(drop=True)
    y_test  = df.loc[split == "test", target_col].reset_index(drop=True)

    # =============================
    # Scaling
    # =============================
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled   = scaler.transform(X_val)

    # =============================
    # 📉 BASELINE MODEL
    # =============================
    baseline_model = xgb.XGBRegressor(random_state=72)
    baseline_model.fit(X_train_scaled, y_train)

    baseline_preds = baseline_model.predict(X_val_scaled)

    baseline_r2 = r2_score(y_val, baseline_preds)
    baseline_rmse = np.sqrt(mean_squared_error(y_val, baseline_preds))
    baseline_rho, _ = spearmanr(y_val, baseline_preds)

    #print("\n--- BASELINE MODEL ---")
    #print(f"R² = {baseline_r2:.3f}")
    #print(f"RMSE = {baseline_rmse:.3f}")
    #print(f"Spearman ρ = {baseline_rho:.3f}")

    # =============================
    # 🔍 OPTUNA
    # =============================
    def objective(trial):

        params = {
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "n_estimators": trial.suggest_int("n_estimators", 100, 500),
            "random_state": 72,
            "verbosity": 0
        }

        model = xgb.XGBRegressor(**params)
        model.fit(X_train_scaled, y_train)

        preds = model.predict(X_val_scaled)
        return r2_score(y_val, preds)

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials)

    print("\nBest params:", study.best_params)
    print(f"Best R² (val): {study.best_value:.3f}")

    # =============================
    # 🔁 FINAL MODEL
    # =============================
    best_model = xgb.XGBRegressor(
        **study.best_params,
        random_state=72
    )

    best_model.fit(X_train_scaled, y_train)

    y_pred = best_model.predict(X_val_scaled)

    r2 = r2_score(y_val, y_pred)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    rho, rho_p = spearmanr(y_val, y_pred)

    #print("\n--- OPTUNA MODEL ---")
    #print(f"R² = {r2:.3f}")
    #print(f"RMSE = {rmse:.3f}")
    #print(f"Spearman ρ = {rho:.3f}")

    #print("\n--- IMPROVEMENT ---")
    #print(f"ΔR²   = {r2 - baseline_r2:.3f}")
    #print(f"ΔRMSE = {baseline_rmse - rmse:.3f}")
    #print(f"Δρ    = {rho - baseline_rho:.3f}")

    print("\n--- OPTIMISATION ---")
    print(f"original R² = {baseline_r2:.3f}")
    print(f"optimal  R² = {r2:.3f}")
    print(f"diff.   ΔR² = {r2 - baseline_r2:.3f}")
    
    return {
        "model": best_model,
        "scaler": scaler,
        "features": feature_cols,
        "best_params": study.best_params
    }

In [ ]:
def ensemble_predict(fold_models, df, split_col):

    test_mask = df[split_col] == "test"

    preds_all = []

    for model_dict in fold_models:

        model  = model_dict["model"]
        scaler = model_dict["scaler"]
        feats  = model_dict["features"]

        X_test = df.loc[test_mask, feats]
        X_test_scaled = scaler.transform(X_test)

        preds = model.predict(X_test_scaled)
        preds_all.append(preds)

    return np.mean(preds_all, axis=0)

# Extract and Create Dataset

In [ ]:
# Suppress RDKit and Python warnings
RDLogger.DisableLog('rdApp.*')
warnings.filterwarnings("ignore")

# ===============================================
# 1️. Load and Clean Data
# ===============================================
fname = "../data/SurfPro-MD.csv"
df = pd.read_csv(fname)

print(f"📄 Loaded {len(df)} rows from {fname}")

# Drop rows with missing SMILES
df = df.dropna(subset=["SMILES"]).copy()

# ===============================================
# 2️. Canonicalize SMILES (and keep largest fragment)
# ===============================================
def canonicalize_smiles(smiles, keep_largest=True):
    """Canonicalize SMILES and optionally keep only the largest fragment."""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        if keep_largest and '.' in smiles:
            # Split fragments and keep the one with most heavy atoms
            frags = Chem.GetMolFrags(mol, asMols=True)
            mol = max(frags, key=lambda m: m.GetNumHeavyAtoms())
        return Chem.MolToSmiles(mol, canonical=True)
    except Exception:
        return None

tqdm.pandas(desc="Canonicalizing SMILES")
df["SMILES_canonical"] = df["SMILES"].progress_apply(canonicalize_smiles)

# Drop invalid molecules
df = df.dropna(subset=["SMILES_canonical"]).reset_index(drop=True)

# Report duplicate canonical SMILES
n_total = len(df)
n_unique = df["SMILES_canonical"].nunique()
n_duplicates = n_total - n_unique
print(f"🧬 Found {n_duplicates} duplicate molecules based on canonical SMILES.")

# Drop duplicates (keep first occurrence)
df = df.drop_duplicates(subset="SMILES_canonical", keep="first").reset_index(drop=True)

print(f"✅ After cleaning: {len(df)} unique, valid molecules")

# ===============================================
# 3️. Prepare Descriptor Calculator
# ===============================================
descriptor_names = [desc_name for desc_name, _ in Descriptors._descList]
calculator = MoleculeDescriptors.MolecularDescriptorCalculator(descriptor_names)

# ===============================================
# 4️. Compute Descriptors
# ===============================================
def compute_rdkit_descriptors(smiles):
    """Compute RDKit descriptors for a single SMILES string."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return {f"rdkit-{name}": np.nan for name in descriptor_names}
    try:
        values = calculator.CalcDescriptors(mol)
        return {f"rdkit-{name}": val for name, val in zip(descriptor_names, values)}
    except Exception:
        return {f"rdkit-{name}": np.nan for name in descriptor_names}

print("🧪 Computing RDKit molecular descriptors...")
tqdm.pandas(desc="Descriptors")
rdkit_descs = df["SMILES_canonical"].progress_apply(compute_rdkit_descriptors)

rdkit_df = pd.DataFrame.from_records(rdkit_descs)
assert len(rdkit_df) == len(df), "❌ Descriptor DataFrame length mismatch!"

# ===============================================
# 5️. Merge and Save
# ===============================================
df_final = pd.concat([df, rdkit_df], axis=1)
df_final["SMILES"] = df_final["SMILES_canonical"]
df_final.reset_index(drop=True, inplace=True)
df=df_final
df["Gamma_max"] = df["Gamma_max"] * 10**6

print(f"✅ Final dataset: {len(df_final)} molecules, {len(rdkit_df.columns)} RDKit descriptors added.")

# Creating Test/Train Split


In [ ]:
def create_cluster_splits(df_input, target_col, ):

    df_t = df_input.dropna(subset=[target_col, "cluster"]).copy()
    df_t["cluster"] = df_t["cluster"].astype(int)

    total_mols = len(df_t)

    
    min_test_size = int(total_mols * MIN_TEST_FRACTION)
    max_test_size = int(total_mols * MAX_TEST_FRACTION)

    cluster_sizes_all = df_t.groupby("cluster").size().to_dict()
    clusters = np.array(list(cluster_sizes_all.keys()))

    for test_split_id in range(N_TEST_SPLITS):

        rng = np.random.RandomState(RANDOM_STATE + test_split_id)

        # ============================
        # TEST SPLIT (cluster-level)
        # ============================
        test_clusters, _ = sample_test_clusters(
            clusters=clusters,
            cluster_sizes=cluster_sizes_all,
            min_size=min_test_size,
            max_size=max_test_size,
            rng=rng,
            max_rejects=MAX_REJECT_TRIES
        )
        
        col_test = f"{target_col}_test_split_{test_split_id}"

        df_t[col_test] = "train_val"
        df_t.loc[df_t["cluster"].isin(test_clusters), col_test] = "test"
    
        # ============================
        # REMAINING DATA (train + val)
        # ============================
        remaining_df = df_t[df_t[col_test] != "test"]
        remaining_clusters = np.array(remaining_df["cluster"].unique())
        remaining_sizes = remaining_df.groupby("cluster").size().to_dict()

        remaining_mols = len(remaining_df)
        
        min_val_size = int(remaining_mols * MIN_VAL_FRACTION)
        max_val_size = int(remaining_mols * MAX_VAL_FRACTION)

        # ============================
        # VAL SPLITS (cluster-level)
        # ============================
        for fold_id in range(N_FOLDS):

            rng_fold = np.random.RandomState(
                RANDOM_STATE + test_split_id * 100 + fold_id
            )

            val_clusters, _ = sample_test_clusters(
                clusters=remaining_clusters,
                cluster_sizes=remaining_sizes,
                min_size=min_val_size,
                max_size=max_val_size,
                rng=rng_fold,
                max_rejects=MAX_REJECT_TRIES
            )

            train_clusters = set(remaining_clusters) - set(val_clusters)

            col = f"{target_col}_split_t{test_split_id}_f{fold_id}"

            df_t[col] = "unused"

            df_t.loc[df_t["cluster"].isin(train_clusters), col] = "train"
            df_t.loc[df_t["cluster"].isin(val_clusters), col] = "val"
            df_t.loc[df_t[col_test] == "test", col] = "test"
    
    return df_t

In [ ]:
# ============================
# Suppress RDKit warnings
# ============================
RDLogger.DisableLog('rdApp.*')
warnings.filterwarnings("ignore", category=DeprecationWarning)

def sample_test_clusters(
    clusters,
    cluster_sizes,
    min_size,
    max_size,
    rng,
    max_rejects=5
):
    available_clusters = set(clusters)
    
    test_clusters = set()
    test_count = 0
    reject_streak = 0

    while available_clusters:

        c = rng.choice(list(available_clusters))
        available_clusters.remove(c)

        new_size = test_count + cluster_sizes[c]

        # Case 1: still too small → always accept
        if new_size < min_size:
            test_clusters.add(c)
            test_count = new_size
            reject_streak = 0
            continue

        # Case 2: within acceptable range → accept and finish
        if min_size <= new_size <= max_size:
            test_clusters.add(c)
            test_count = new_size
            break

        # Case 3: too large → reject
        reject_streak += 1

        if reject_streak >= max_rejects:
            print("COULD NOT CREATE SET")
            # Give up and accept current test set
            break

    return test_clusters, test_count


# ============================
# 1. Convert SMILES to Mol objects
# ============================
df["mol"] = df["SMILES"].apply(Chem.MolFromSmiles)

df_valid = df[df["mol"].notnull()].copy()
# ============================
# 2. Generate Morgan fingerprints
# ============================
def get_fingerprint(mol, radius=2, n_bits=1024):
    if mol is None:
        return None
    return AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)

fps = [get_fingerprint(m) for m in df_valid["mol"]]
df_valid["fingerprint"] = fps
nfps = len(fps)

print(f"Generated {nfps} valid fingerprints.")

# ============================
# 3. Compute pairwise Tanimoto distances
# ============================
dists = []
for i in range(1, nfps):
    sims = DataStructs.BulkTanimotoSimilarity(fps[i], fps[:i])
    dists.extend([1 - x for x in sims])

# ============================
# 4. Butina clustering
# ============================
dist_thresh = 0.6
clusters = Butina.ClusterData(dists, nfps, distThresh=dist_thresh, isDistData=True)

cluster_id = np.zeros(nfps, dtype=int)
for i, cluster in enumerate(clusters):
    for idx in cluster:
        cluster_id[idx] = i

df_valid["cluster"] = cluster_id
df_valid["cluster"] = df_valid["cluster"].astype("Int64") 

print(f"Found {len(clusters)} clusters.")

# ============================
# 5. t-SNE (optional, unchanged)
# ============================
fps_np = np.zeros((nfps, 1024), dtype=int)
for i, fp in enumerate(fps):
    DataStructs.ConvertToNumpyArray(fp, fps_np[i])

tsne = TSNE(n_components=2, perplexity=20, random_state=3, metric="cosine")
tsne_result = tsne.fit_transform(fps_np)

df_valid["tsne_1"] = tsne_result[:, 0]
df_valid["tsne_2"] = tsne_result[:, 1]

# ============================
# 6. Merge back into main df
# ============================
df["cluster"] = np.nan
df.loc[df_valid.index, "cluster"] = df_valid["cluster"].values
df.loc[df_valid.index, "tsne_1"] = df_valid["tsne_1"].values
df.loc[df_valid.index, "tsne_2"] = df_valid["tsne_2"].values

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

plt.rcParams.update({
    "font.size": 14,
    "axes.spines.top": False,
    "axes.spines.right": False
})

# ============================================================
# FIGURE LAYOUT (TOP PANEL RESERVED)
# ============================================================
fig = plt.figure(figsize=(11, 5))

gs = fig.add_gridspec(
    nrows=1,
    ncols=4,
    width_ratios=[1, 0.2, 1.0, 0.05],  # last column = colorbar
    wspace=0.1
)

gs_left = gs[0, 0].subgridspec(
    3, 3,
    wspace=0.25,
    hspace=0.5
)

ax_corr = fig.add_subplot(gs[0, 2])
ax_cbar = fig.add_subplot(gs[0, 3])

# ============================================================
# PANEL B1 — DISTRIBUTIONS (IMPROVED)
# ============================================================

TARGETS = [
    "pCMC", "D_MOL", "D_SOL",
    "surface_tension_avg", "viscosity", "AW_ST_CMC",
    "Gamma_max", "Area_min", "pC20"
]

target_labels = {
    "pCMC": "pCMC",
    "AW_ST_CMC": r"$\gamma_{\mathrm{CMC}}$",
    "Gamma_max": r"$\Gamma_{\max}$",
    "pC20": "pC$_{20}$",
    "Pi_CMC": r"$\Pi_{\mathrm{CMC}}$",
    "Area_min": r"A$_{\min}$",
    "viscosity": r"$\eta$",
    "D_MOL": r"D$_{\mathrm{MOL}}$",
    "D_SOL": r"D$_{\mathrm{SOL}}$",
    "surface_tension_avg": r"$\gamma$",
}

unit_labels = {
    "pCMC": "pCMC",
    "AW_ST_CMC": r"$\gamma_{\mathrm{CMC}}$ [mN m$^{-1}$]",
    "Gamma_max": r"$\Gamma_{\max}$ [$\mu$M m$^{-2}$]",
    "pC20": "pC$_{20}$",
    "Pi_CMC": r"$\Pi_{\mathrm{CMC}}$ [mN m$^{-1}$]",
    "Area_min": r"A$_{\min}$ [nm$^2$]",
    "viscosity": r"$\eta$ [mP·s]",
    "D_MOL": r"D$_\mathrm{MOL}$ [m$^2$s$^{-1}$]",
    "D_SOL": r"D$_\mathrm{SOL}$ [m$^2$s$^{-1}$]",
    "surface_tension_avg": r"$\gamma$ [mN m$^{-1}$]",
}

axes_hist = []

colors = cm.viridis(np.linspace(0.15, 0.85, len(TARGETS)))

for i, target in enumerate(TARGETS):

    ax = fig.add_subplot(gs_left[i // 3, i % 3])
    axes_hist.append(ax)

    values = df[target].dropna().values

    ax.hist(
        values,
        bins=25,
        density=True,
        alpha=0.75,
        color=colors[0]
    )

    # cleaner axes
    ax.set_yticklabels([])
    ax.set_yticks([])
    #ax.set_aspect('equal', adjustable='box')
    ymin, ymax = ax.get_ylim()
    ax.set_ylim(ymin, ymax * 1.25)  # add ~15% headroom
    
    ax.set_xlabel("")
    ax.text(
        0.05, 1.05,
        unit_labels[target],
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=14
    )
    
# ============================================================
# PANEL B2 — CORRELATION MATRIX (CLEANED)
# ============================================================

TARGETS_CORR = [
    "pCMC", "D_MOL", "D_SOL",
    "surface_tension_avg", "viscosity",
    "AW_ST_CMC", "Gamma_max",
    "Area_min", "pC20", "Pi_CMC"
]

df_corr = df[TARGETS_CORR]

labels = [
    "pCMC",
    r"$\gamma_{\mathrm{CMC}}$",
    r"$\Gamma_{\max}$",
    "pC$_{20}$",
    r"$\Pi_{\mathrm{CMC}}$",
    r"A$_{\min}$",
    r"$\eta$",
    r"D$_{\mathrm{MOL}}$",
    r"D$_{\mathrm{SOL}}$",
    r"$\gamma$"
]

corr = df_corr.corr()

ax_corr.imshow(corr.values, cmap="viridis", vmin=-1, vmax=1)

ax_corr.set_xticks(np.arange(len(TARGETS_CORR)))
ax_corr.set_yticks(np.arange(len(TARGETS_CORR)))

ax_corr.set_xticklabels(labels, rotation=45, ha="right", fontsize=14)
ax_corr.set_yticklabels(labels, fontsize=14)

# annotate only strong correlations
for i in range(len(TARGETS_CORR)):
    for j in range(len(TARGETS_CORR)):
        if corr.values[i,j]<=-0.46:
            ax_corr.text(j, i, f"{corr.values[i,j]:.1f}",
                ha="center", va="center", fontsize=11, color="white")
        else:
            ax_corr.text(j, i, f"{corr.values[i,j]:.1f}",
                    ha="center", va="center", fontsize=11)

im = ax_corr.imshow(corr.values, cmap="viridis", vmin=-1, vmax=1)
cbar = fig.colorbar(im, cax=ax_cbar)
cbar.set_label("Pearson correlation", fontsize=11)
cbar.ax.tick_params(labelsize=11)

ax_corr.grid(False)
ax_corr.set_aspect('equal')
# ============================================================
# LAYOUT FINALIZATION
# ============================================================
fig.text(0.1, 0.85, "a", fontsize=14, fontweight="bold")
fig.text(0.50, 0.85, "b", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.savefig("figure_data.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
def filter_outliers(df, target):
    
    n_before = len(df)
    if target == "surface_tension_avg":
        df= df[
            (df["surface_tension_avg"].notnull()) &
            (df["surface_tension_avg"] >= 250)
        ]
    
    elif target == "viscosity":
        df= df[
            (df["viscosity"].notnull()) &
            (df["viscosity"] <= 0.003)
        ]
    elif target == "AW_ST_CMC":
        df= df[
            (df["AW_ST_CMC"].notnull()) &
            (df["AW_ST_CMC"] <= 52)
        ]
    elif target == "Gamma_max":
        df= df[
            (df["Gamma_max"].notnull()) &
            (df["Gamma_max"] <= 6) #float(6e-6))
        ]
    elif target == "Area_min":
        df= df[
            (df["Area_min"].notnull()) &
            (df["Area_min"] <= 4.2)
        ]
    elif target == "pC20":
        df= df[
            (df["pC20"].notnull()) &
            (df["pC20"] >= 1.8)
        ]
    elif target == "D_MOL":
        df= df[
            (df["D_MOL"].notnull()) &
            (df["D_MOL"] <= 0.8)
        ]
   
    n_after = len(df)

    print(f"Filtering Target: {target}")
    print(f"Rows before filtering: {n_before}")
    print(f"Rows after filtering:  {n_after}")
    print(f"Rows removed:         {n_before - n_after}")
    print(f"Fraction removed:     {(n_before - n_after)/n_before:.2%}")

    return df

def get_fold_group(fold_id, n_folds_per_model):
    return fold_id // n_folds_per_model

In [ ]:
TARGETS = [ "pCMC", "AW_ST_CMC", "Gamma_max", "Area_min", "pC20", "D_MOL", "D_SOL", "surface_tension_avg", "viscosity"]
N_FOLDS = 5
N_TEST_SPLITS = 5
MIN_TEST_FRACTION = 0.07
MAX_TEST_FRACTION = 0.11
MAX_REJECT_TRIES = 5
MIN_VAL_FRACTION = 0.12
MAX_VAL_FRACTION = 0.18     
RANDOM_STATE=3

total_mols = len(df)
min_test_size = int(total_mols * MIN_TEST_FRACTION)
max_test_size = int(total_mols * MAX_TEST_FRACTION)
min_val_size = int(total_mols * MIN_VAL_FRACTION)
max_val_size = int(total_mols * MAX_VAL_FRACTION)

optuna.logging.set_verbosity(optuna.logging.WARNING)


results = {}

for target in TARGETS:
    print("\n")
    print(f"Working on {target}")
    df_target = create_cluster_splits(df, target)
    df_target = filter_outliers(df_target, target)
    

    rdkit_cols = [c for c in df_target.columns if c.startswith("rdkit-")]
    features_all = rdkit_cols
    y_all = df_target[[target]]

    target_results = []

    for test_id in range(N_TEST_SPLITS):
    
        print(f"\n### TEST SPLIT {test_id}")
    
        fold_models = []
    
        for fold in range(N_FOLDS):
    
            split_col = f"{target}_split_t{test_id}_f{fold}"
            split = df_target[split_col]
    
            model = optimise(
                df=df_target,
                feature_cols=features_all,
                target_col=target,
                split=split,
                outname=target,
                draw_plots=False
            )
    
            fold_models.append(model)

        #==========================  
        # EVALUATE
        #==========================
        split_col0 = f"{target}_split_t{test_id}_f0"
        ensemble_preds = ensemble_predict(
            fold_models,
            df_target,
            split_col0
        )
        
        # ============================
        # Ground truth
        # ============================
        test_mask = df_target[split_col0] == "test"
        y_test = df_target.loc[test_mask, target].values
        
        # ============================
        # Evaluate ENSEMBLE (this is your final model now)
        # ============================
        ensemble_r2_s = r2_score(y_test, ensemble_preds)
        ensemble_rmse_s = np.sqrt(mean_squared_error(y_test, ensemble_preds))
        ensemble_rho_s, _ = spearmanr(y_test, ensemble_preds)
        ensemble_pcc_s, _ = pearsonr(y_test, ensemble_preds)
        
        print(f"ENSEMBLE R² = {ensemble_r2_s:.3f}")
        # print(f"ENSEMBLE PCC = {ensemble_pcc_s:.3f}")
        # print(f"ENSEMBLE RMSE = {ensemble_rmse_s:.3f}")
        # print(f"ENSEMBLE Spearman ρ = {ensemble_rho_s:.3f}")
        
        # ============================
        # Store results
        # ============================
        test_indices = df_target.index[test_mask]
        
        target_results.append({
            "target": target,               
            "test_id": test_id,
        
            # metrics
            "rmse_ensemble": ensemble_rmse_s,
            "r2_ensemble": ensemble_r2_s,
            "rho_ensemble": ensemble_rho_s,
            "pcc_ensemble": ensemble_pcc_s,
        
            # models
            "fold_models": fold_models,
        
            # data traceability
            "test_indices": test_indices,
            "split_column": split_col0,
        
            # reproducibility
            "feature_cols": features_all,
            "y_test": y_test,
            "y_pred": ensemble_preds,
            "test_mask": test_mask,
        })

        results[f"{target}_{test_id}"] = target_results
        
        with open("models.pkl", "wb") as f:
    
            pickle.dump(results, f)

In [ ]:
from collections import defaultdict

grouped = defaultdict(list)

for run_list in results.values():        # this is a LIST
    for r in run_list:                   # r is the actual dict
        grouped[r["target"]].append(r)

metrics_summary = {}

for target, res_list in grouped.items():

    r2_ens   = [r["r2_ensemble"]   for r in res_list]
    rho_ens  = [r["rho_ensemble"]  for r in res_list]
    rmse_ens = [r["rmse_ensemble"] for r in res_list]
    pcc_ens  = [r["pcc_ensemble"]  for r in res_list]

    metrics_summary[target] = {
        "r2_mean":  np.mean(r2_ens),
        "r2_std":   np.std(r2_ens),

        "rho_mean": np.mean(rho_ens),
        "rho_std":  np.std(rho_ens),

        "rmse_mean": np.mean(rmse_ens),
        "rmse_std":  np.std(rmse_ens),

        "pcc_mean":  np.mean(pcc_ens),
        "pcc_std":   np.std(pcc_ens),
    }

targets = list(metrics_summary.keys())

# R²
r2_means = [metrics_summary[t]["r2_mean"] for t in targets]
r2_errs  = [metrics_summary[t]["r2_std"]  for t in targets]

# Spearman ρ
rho_means = [metrics_summary[t]["rho_mean"] for t in targets]
rho_errs  = [metrics_summary[t]["rho_std"]  for t in targets]

# RMSE
rmse_means = [metrics_summary[t]["rmse_mean"] for t in targets]
rmse_errs  = [metrics_summary[t]["rmse_std"]  for t in targets]

x = np.arange(len(targets))
width = 0.35

plt.figure()

plt.bar(x - width/2, r2_means, width, yerr=r2_errs, label="Ensemble R²")
plt.bar(x + width/2, rho_means, width, yerr=rho_errs, label="Ensemble Spearman ρ")

plt.legend()
plt.xticks(x, targets, rotation=45, ha="right")
plt.ylabel("Score")
plt.tight_layout()
#plt.savefig()
plt.show()

for target in metrics_summary:

    r2_mean = metrics_summary[target]["r2_mean"]
    r2_std  = metrics_summary[target]["r2_std"]

    rho_mean = metrics_summary[target]["rho_mean"]
    rho_std  = metrics_summary[target]["rho_std"]

    rmse_mean = metrics_summary[target]["rmse_mean"]
    rmse_std  = metrics_summary[target]["rmse_std"]
        

    print(
        f"{target} & "
        f"{r2_mean:.3f} $\\pm$ {r2_std:.3f} & "
        f"{rho_mean:.3f} $\\pm$ {rho_std:.3f} & "
        f"{rmse_mean:.3f} $\\pm$ {rmse_std:.3f} \\\\"
    )

In [ ]:
def ensemble_predict(models, X):
    preds = []
    for m in models:
        preds.append(m["model"].predict(X))
    return np.mean(preds, axis=0)

def build_df_target(df, target):
    df_target = create_cluster_splits(df, target)
    df_target = filter_outliers(df_target, target)
    return df_target

def ensemble_permutation_importance(entry, df, n_repeats=10, random_state=0):

    np.random.seed(random_state)

    if isinstance(entry, list):
        entry = entry[0]

    target = entry["target"]

    # 🔥 RECONSTRUCT EXACT TRAINING DATA
    df_target = build_df_target(df, target)

    models = entry["fold_models"]
    feature_cols = entry["feature_cols"]

    # IMPORTANT: use indices relative to df_target
    test_mask = df_target[entry["split_column"]] == "test"

    X_test = df_target.loc[test_mask, feature_cols]
    y_test = df_target.loc[test_mask, target]

    base_pred = ensemble_predict(models, X_test)
    base_score = r2_score(y_test, base_pred)

    importances = {f: [] for f in feature_cols}

    for feature in feature_cols:
        for _ in range(n_repeats):

            X_perm = X_test.copy()
            X_perm[feature] = np.random.permutation(X_perm[feature].values)

            perm_pred = ensemble_predict(models, X_perm)
            perm_score = r2_score(y_test, perm_pred)

            importances[feature].append(base_score - perm_score)

    return pd.DataFrame({
        "feature": list(importances.keys()),
        "importance_mean": [np.mean(v) for v in importances.values()],
        "importance_std": [np.std(v) for v in importances.values()],
    }).sort_values("importance_mean", ascending=False)

importance_results = {}


for key, entry in results.items():

    print(f"Processing {key}")

    df_importance = ensemble_permutation_importance(
        entry=entry,
        df=df,   # or your full df
        n_repeats=5
    )

    importance_results[key] = df_importance



In [ ]:
from collections import defaultdict

agg = defaultdict(list)

for key, df_imp in importance_results.items():
    target = results[key][0]["target"]

    for _, row in df_imp.iterrows():
        agg[(target, row["feature"])].append(row["importance_mean"])

summary = []

for (target, feature), vals in agg.items():
    summary.append({
        "target": target,
        "feature": feature,
        "importance_mean": np.mean(vals),
        "importance_std": np.std(vals),
    })

In [ ]:
TOP_K = 10

from collections import defaultdict
import matplotlib.pyplot as plt

# ----------------------------
# target -> list of top features
# ----------------------------
top_features_per_target = defaultdict(list)

for key, df_imp in importance_results.items():

    target = results[key][0]["target"]

    top_feats = (
        df_imp.sort_values("importance_mean", ascending=False)
              .head(TOP_K)["feature"]
              .tolist()
    )

    top_features_per_target[target].extend(top_feats)

# ----------------------------
# collect all unique features
# ----------------------------
all_top_features = set()

for feats in top_features_per_target.values():
    all_top_features.update(feats)

# ----------------------------
# count in how many targets each feature appears
# ----------------------------
feature_counts = defaultdict(int)

for feature in all_top_features:
    for target, feats in top_features_per_target.items():
        if feature in feats:
            feature_counts[feature] += 1

# ----------------------------
# histogram: count how many features appear in k targets
# ----------------------------
hist = defaultdict(int)

for feature, count in feature_counts.items():
    hist[count] += 1

# ----------------------------
# plot
# ----------------------------
x_vals = sorted(hist.keys(), reverse=True)
y_vals = [hist[x] for x in x_vals]

plt.figure()

plt.bar(x_vals, y_vals)

plt.xlabel("Number of targets (out of 9)")
plt.ylabel("Number of features")
plt.title("Feature overlap across top-10 features per target")

plt.xticks(x_vals)

plt.tight_layout()
plt.show()


In [ ]:
from collections import defaultdict, Counter

# ----------------------------
# collect top-K features per target
# ----------------------------
top_features_per_target = defaultdict(list)

for key, df_imp in importance_results.items():

    target = results[key][0]["target"]

    top_feats = (
        df_imp.sort_values("importance_mean", ascending=False)
              .head(TOP_K)["feature"]
              .tolist()
    )

    top_features_per_target[target].extend(top_feats)

# ----------------------------
# build feature -> set(targets)
# ----------------------------
feature_to_targets = defaultdict(set)

for target, feats in top_features_per_target.items():
    for f in set(feats):  # set avoids double counting within a target
        feature_to_targets[f].add(target)

# ----------------------------
# count occurrences
# ----------------------------
feature_counts = {
    f: len(tset) for f, tset in feature_to_targets.items()
}

# ----------------------------
# find top 2 universality levels
# ----------------------------
unique_counts = sorted(feature_counts.values(), reverse=True)
top_2_levels = sorted(set(unique_counts[:2]), reverse=True)

# ----------------------------
# filter features
# ----------------------------
important_features = {
    f: c for f, c in feature_counts.items()
    if c in top_2_levels
}

# ----------------------------
# print results (NOW WITH TARGETS)
# ----------------------------
print("\nMost important cross-target features (top 2 universality levels):\n")

for f, c in sorted(important_features.items(), key=lambda x: -x[1]):
    
    targets_here = sorted(feature_to_targets[f])
    targets_str = ", ".join(targets_here)

    print(f"{f:60s}  -> {c} targets")
    print(f"{'':60s}     [{targets_str}]\n")

# MAKE FINAL FIGURES

In [ ]:
def read_xvg(fname, quantity):
        values=[]
        eqsteps = 100
        if fname[-3:] != "xvg":
            print( "The provided file is not in .xvg format.")
            sys.exit(0)
        with open(fname, 'r') as fid:
            ll = fid.readlines()
        M = len(ll[-1].split())
        labels = [""]*M
        labels[0] = "Time (ps)"
        #print ("Found",M,"columns:")
        #print ('  ',"[1] Time")
        ALL = []
        icol=-1
        i=1
        for l in ll:
          if l.startswith("@ s"):
              label = ' '.join(l.split()[3:])[1:-1]
              #print ('  ','[%d]'%(i+1),label)
              labels[i] = label
              if label.strip()==quantity.strip():
                  icol=i
              i += 1
          if l[0] in ['#','@']:
            continue
          s = l.split()
          ALL.append([float(x) for x in s])
        
        if icol==-1:
            print("QUANTITY HAS NOT BEEN COMPUTED IN " + str(fname))
            return 0
        ALL = np.asarray(ALL)
        #print( '  ',"Data array dimensions:",ALL.shape[0],'x',ALL.shape[1])
        ALL=ALL.transpose()
        #print( '  ',"Data array dimensions:",ALL.shape[0],'x',ALL.shape[1])
        average = np.average(ALL[icol,eq_steps:])
        std     = np.std(ALL[icol,eq_steps:])
        #print("viscosity of " + molecule + " is " + str(viscosity))

        return [average, std]

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.gridspec as gridspec
from matplotlib.ticker import FuncFormatter
from matplotlib.lines import Line2D
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset

plt.rcParams.update({'font.size': 14})


target_labels = {
    "pCMC": "pCMC",
    "AW_ST_CMC": r"$\gamma_{\mathrm{CMC}}$",
    "Gamma_max": r"$\Gamma_{\max}$",
    "pC20": "pC$_{20}$",
    "Pi_CMC": r"$\Pi_{\mathrm{CMC}}$",
    "Area_min": r"A$_{\min}$",
    "viscosity": r"$\eta$",
    "D_MOL": r"D$_{\mathrm{MOL}}$",
    "D_SOL": r"D$_{\mathrm{SOL}}$",
    "surface_tension_avg": r"$\gamma$",
}
# ----------------------------
# helper: viridis colors
# ----------------------------
def get_colors(n):
    cmap = cm.get_cmap("viridis")
    return [cmap(i/(n-1)) for i in range(n)]

# ----------------------------
# Figure layout
# ----------------------------

fig = plt.figure(figsize=(11, 5))

gsAB = gridspec.GridSpec(
    nrows=1,
    ncols=2,
    wspace=0.2  
)

ax_tension = fig.add_subplot(gsAB[0, 0])
ax_visc = fig.add_subplot(gsAB[0, 1])

axins = inset_axes(
    ax_tension,
    width="100%",
    height="100%",
    bbox_to_anchor=(0.5, 0.3, 0.4, 0.4),  # (x, y, w, h)
    bbox_transform=ax_tension.transAxes,
    loc="lower left"
)

# zoom region (tune these!)
axins.set_xlim(-0.02, 0.2)
axins.set_ylim(0.6, 1.1)

axins.tick_params(labelsize=8)

# draw box on main plot
#mark_inset(ax_tension, axins, loc1=2, loc2=4, fc="none", ec="0.5")

# ============================================================
# PANEL A1 — Relative Surface Tension (CORRECTED)
# ============================================================

all_nmol = [150, 300, 450, 1500, 2500]
all_mol = ["methanol", "ethanol", "propanol"]

colors = get_colors(len(all_mol))

for icolor, mol in enumerate(all_mol):

    df_ref = pd.read_csv(f"/home/ricbec/PFAS-project/reference/{mol}_tension.csv")

    # store per-concentration replica arrays
    replica_data = {}

    for nmol in all_nmol:

        runs = []

        for replica in range(19):
            fname = f"/home/ricbec/PFAS-project/data/tension-alcohols/{mol}_{nmol}/ST_{str(replica).zfill(2)}/potential.xvg"
            avg_single, _ = read_xvg(fname, "#Surf*SurfTen")
            runs.append(avg_single / 20)   # scale here already

        replica_data[nmol] = np.array(runs)

    # reference concentration (lowest nmol)
    nmol0 = all_nmol[0]
    gamma0_replicas = replica_data[nmol0]

    rows = []

    for nmol in all_nmol:

        gamma_replicas = replica_data[nmol]

        # 🔥 per-replica normalization
        rel = gamma_replicas / gamma0_replicas

        rows.append({
            "c": nmol / 15000,
            "rel_tension_mean": np.mean(rel),
            "rel_tension_std": np.std(rel)
        })

    df = pd.DataFrame(rows).sort_values("c")

    # reference (unchanged)
    df_ref = df_ref.sort_values("mol_fraction")
    gamma0_ref = df_ref["surface_tension"].iloc[0]
    df_ref["rel_tension"] = df_ref["surface_tension"] / gamma0_ref

    # ----------------------------
    # plotting
    # ----------------------------
    ax_tension.errorbar(
        df["c"],
        df["rel_tension_mean"],
        yerr=df["rel_tension_std"],
        color=colors[icolor],
        marker='o',
        ms=0.1,
        lw=1.5,
        capsize=3,
        label=f"{mol} comp."
    )

    ax_tension.plot(
        df_ref["mol_fraction"],
        df_ref["rel_tension"],
        ls="--",
        marker='o',
        ms=4,
        color=colors[icolor],
        lw=1.5,
        label=f"{mol} exp."
    )
    axins.errorbar(
        df["c"],
        df["rel_tension_mean"],
        yerr=df["rel_tension_std"],
        color=colors[icolor],
        lw=1.2
    )

    axins.plot(
        df_ref["mol_fraction"],
        df_ref["rel_tension"],
        ls="--",
        color=colors[icolor],
        lw=1.2
    )


ax_tension.set_xlabel("Mol Fraction")
ax_tension.set_ylabel("Relative surface tension")
ax_tension.set_ylim([0.3,1.2])

# --- style legend (exp vs comp) ---
style_handles = [
    Line2D([0], [0], color="gray", lw=1.5, ls="-", label="comp."),
    Line2D([0], [0], color="gray", lw=1.5, ls="--", label="exp.")
]

# --- molecule legend (colors only) ---
mol_handles = [
    Line2D([0], [0], color=colors[i], lw=2, label=mol)
    for i, mol in enumerate(all_mol)
]

# combine
handles = style_handles + mol_handles

ax_tension.legend(handles=handles, frameon=False)






# ============================================================
# PANEL A2 — Relative Viscosity
# ============================================================
all_nmol = [10, 30, 50, 100, 150, 200]
all_mol = ["formic", "acetic", "propionic", "valeric"]

colors = get_colors(len(all_mol))

df_ref = pd.read_csv("/home/ricbec/PFAS-project/reference/acid_reference.csv")
df_ref["c"] = df_ref["c*100"] / 100

# ----------------------------
# baseline (pure solvent)
# ----------------------------
base_data = pd.DataFrame()

for mol in all_mol:
    all_runs = []

    for replica in range(10):
        fname = f"/home/ricbec/PFAS-project/data/viscosities/{mol}_acid_0/ST_{str(replica).zfill(2)}/visco.xvg"
        avg_single, _ = read_xvg(fname, "1/Viscosity")
        all_runs.append(1 / avg_single)

    base_data = pd.concat([base_data, pd.DataFrame([{
        "acid": mol,
        "viscosity": np.mean(all_runs)
    }])], ignore_index=True)

# ----------------------------
# concentration-dependent
# ----------------------------
df = pd.DataFrame()

for mol in all_mol:
    for nmol in all_nmol:

        all_runs = []
        all_c = []

        for replica in range(10):

            fname = f"/home/ricbec/PFAS-project/data/viscosities/{mol}_acid_{nmol}/ST_{str(replica).zfill(2)}/visco.xvg"
            avg_single, _ = read_xvg(fname, "1/Viscosity")
            all_runs.append(1 / avg_single)

            with open(f"/home/ricbec/PFAS-project/data/viscosities/{mol}_acid_{nmol}/ST_{str(replica).zfill(2)}/production.gro") as f:
                boxsize = float(f.readlines()[-1].split()[0])

            c_nm = nmol / (boxsize ** 3)
            c = 10 / 6.023 * c_nm
            all_c.append(c)

        visco = np.mean(all_runs)
        std = np.std(all_runs)

        base_visco = base_data[base_data["acid"] == mol]["viscosity"].values[0]
        rel_visco = 1 + (visco / base_visco - 1) * 72 / 50

        df = pd.concat([df, pd.DataFrame([{
        "c": np.mean(all_c),
        "acid": mol,
        "rel_viscosity": rel_visco,
        "error": std / base_visco   # normalized error
        }])], ignore_index=True)

# ----------------------------
# plotting
# ----------------------------
for icolor, mol in enumerate(all_mol):

    subset = df[df["acid"] == mol].sort_values("c")

    subset_ref = df_ref[df_ref["acid"] == mol].sort_values("c")

    visco0_ref = subset_ref["viscosity"].iloc[0]
    rel_visco_ref = subset_ref["viscosity"] / visco0_ref

    ax_visc.errorbar(
        subset["c"],
        subset["rel_viscosity"],
        yerr=subset["error"],
        color=colors[icolor],
        marker='o',
        ms=0.1,
        lw=1.5,
        capsize=3,
        label=f"{mol} comp."
    )
    
    # reference (line only)
    ax_visc.plot(
        subset_ref["c"],
        rel_visco_ref,
        ls="--",
        marker='o',
        ms=4,
        color=colors[icolor],
        lw=1.5,
        label=f"{mol} exp."
    )

# right-side axis
ax_visc.yaxis.tick_left()
#ax_visc.yaxis.set_label_position("right")


ax_visc.set_xlabel("Concentration [mol/L]")
ax_visc.set_ylabel("Relative viscosity")

# clean middle boundary
ax_tension.spines['left'].set_visible(True)
ax_visc.spines['left'].set_visible(True)

ax_visc.set_ylim([1,1.3499])
ax_tension.set_ylim([0.3,1.1])
ax_visc.tick_params(axis='y', pad=2)
ax_tension.tick_params(axis='y', pad=2)

# --- molecule legend (colors only) ---
mol_handles = [
    Line2D([0], [0], color=colors[i], lw=2, label=mol)
    for i, mol in enumerate(all_mol)
]

# combine
handles = style_handles + mol_handles

ax_visc.legend(handles=handles, frameon=False)


fig.text(0.05, 0.85, "a", fontsize=14, fontweight="bold")
fig.text(0.50, 0.85, "b", fontsize=14, fontweight="bold")

plt.savefig("MD_validation.pdf")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.gridspec as gridspec
from matplotlib.ticker import FuncFormatter
import string
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset
plt.rcParams.update({'font.size': 14})


target_labels = {
    "pCMC": "pCMC",
    "AW_ST_CMC": r"$\gamma_{\mathrm{CMC}}$",
    "Gamma_max": r"$\Gamma_{\max}$",
    "pC20": "pC$_{20}$",
    "Pi_CMC": r"$\Pi_{\mathrm{CMC}}$",
    "Area_min": r"A$_{\min}$",
    "viscosity": r"$\eta$",
    "D_MOL": r"D$_{\mathrm{MOL}}$",
    "D_SOL": r"D$_{\mathrm{SOL}}$",
    "surface_tension_avg": r"$\gamma$",
}
# ----------------------------
# helper: viridis colors
# ----------------------------
def get_colors(n):
    cmap = cm.get_cmap("viridis")
    return [cmap(i/(n-1)) for i in range(n)]

# ----------------------------
# Figure layout
# ----------------------------

fig = plt.figure(figsize=(11, 12))

gs = gridspec.GridSpec(
    nrows=4,
    ncols=3,
    height_ratios=[1, 1, 1, 1],
    hspace=0.35,
    wspace=0.15  
)


ax_metrics = fig.add_subplot(gs[0, :])

# ============================
# PANEL B — ML performance + feature overlap
# ============================

from collections import defaultdict
import numpy as np
import matplotlib.cm as cm

# ============================================================
# LEFT: R² + Spearman ρ
# ============================================================

grouped = defaultdict(list)

for run_list in results.values():
    for r in run_list:
        grouped[r["target"]].append(r)

metrics_summary = {}

for target, res_list in grouped.items():

    r2_ens   = [r["r2_ensemble"]   for r in res_list]
    rho_ens  = [r["rho_ensemble"]  for r in res_list]

    metrics_summary[target] = {
        "r2_mean":  np.mean(r2_ens),
        "r2_std":   np.std(r2_ens),
        "rho_mean": np.mean(rho_ens),
        "rho_std":  np.std(rho_ens),
    }

# optional: sort by R² (makes plot MUCH nicer)
targets = sorted(metrics_summary.keys(), key=lambda t: -metrics_summary[t]["r2_mean"])

x = np.arange(len(targets))
width = 0.35

r2_means = [metrics_summary[t]["r2_mean"] for t in targets]
r2_errs  = [metrics_summary[t]["r2_std"]  for t in targets]

rho_means = [metrics_summary[t]["rho_mean"] for t in targets]
rho_errs  = [metrics_summary[t]["rho_std"]  for t in targets]

ax_metrics.bar(x - width/2, r2_means, width, yerr=r2_errs, label="R²")
ax_metrics.bar(x + width/2, rho_means, width, yerr=rho_errs, label="Spearman ρ")

ax_metrics.set_xticks(x)
ax_metrics.set_xticklabels(
    [target_labels[t] for t in targets],
    rotation=45,
    ha="right"
)
ax_metrics.set_ylabel("Score", labelpad=-10)
ax_metrics.set_ylim([-0.25,1])
ax_metrics.legend(frameon=False)
ax_metrics.spines['top'].set_visible(False)


# ============================================================
# PENAL C: CORRELATION PLOTS
# ============================================================

unit_labels = {
    "pCMC": "pCMC",
    "AW_ST_CMC": r"$\gamma_{\mathrm{CMC}}$ [mN m$^{-1}$]",
    "Gamma_max": r"$\Gamma_{\max}$ [$\mu$mol m$^{-2}$]",
    "pC20": "pC20",
    "Pi_CMC": r"$\Pi_{\mathrm{CMC}}$ [mN m$^{-1}$]",
    "Area_min": r"Area$_{\min}$ [nm$^2$]",
    "viscosity": r"$\eta$ [mP·s]",
    "D_MOL": r"D$_\mathrm{MOL}$ [m$^2$s$^{-1}$]",
    "D_SOL": r"D$_\mathrm{SOL}$ [m$^2$s$^{-1}$]",
    "surface_tension_avg": r"$\gamma$ [mN m$^{-1}$]",
}


plot_data = defaultdict(list)

for run_list in results.values():
    for r in run_list:
        plot_data[r["target"]].append(r)

import matplotlib.cm as cm



alphabet=list(string.ascii_lowercase)

cmap = cm.get_cmap("viridis", 5)

axesC = []

for i, (target, runs) in enumerate(plot_data.items()):

    ax = fig.add_subplot(gs[1 + (i // 3), i % 3])
    axesC.append(ax)
    ax.set_aspect('equal', adjustable='box')

    vmin, vmax = np.inf, -np.inf

    for r in runs:
        test_id = r["test_id"]

        y_true = np.array(r["y_test"])
        y_pred = np.array(r["y_pred"])

        ax.scatter(
            y_true, y_pred,
            s=10,
            alpha=0.7,
            color=cmap(test_id),
            label=f"test {test_id}"
        )

        vmin = min(vmin, y_true.min(), y_pred.min())
        vmax = max(vmax, y_true.max(), y_pred.max())

    # identity line (after loop!)
    ax.plot([vmin, vmax], [vmin, vmax], 'k--', lw=1)

    # metrics
    ax.text(
        0.05, 0.95,
        f"$R^2$ = {metrics_summary[target]['r2_mean']:.2f} ± {metrics_summary[target]['r2_std']:.2f}\n"
        f"$\\rho$ = {metrics_summary[target]['rho_mean']:.2f} ± {metrics_summary[target]['rho_std']:.2f}",
        transform=ax.transAxes,
        va="top",
        fontsize=14
    )

    #ax.set_title(target, fontsize=10)
    ax.set_ylabel(unit_labels[target])
    ax.set_xlabel(unit_labels[target])
    
    if i % 3 == 0:
        ax.set_ylabel(f"Predicted\n{unit_labels[target]}", fontsize=14)
    
    if i >= 6:
        ax.set_xlabel(f"{unit_labels[target]}\nTrue", fontsize=14)

    if target == "viscosity":
        ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: x * 1000))
        ax.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: x * 1000))
    
    ax.text(-0.15, 1.04, alphabet[i+1], transform=ax.transAxes, fontsize=14, fontweight="bold", va="bottom")


plt.subplots_adjust(bottom=0.08, top=0.98, left=0.06, right=0.98)
right_col = [2, 5, 8]

fig.text(0.01, 0.99, "a", fontsize=14, fontweight="bold")



plt.tight_layout()
plt.savefig("full_results.pdf", dpi=300, bbox_inches="tight")
plt.show()